In [6]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point
import numpy as np

In [3]:
data = pd.read_csv('./data/location.csv')

data.head()

,sampno,locno,perno,loctype,state,country,state_fips,county_fips,tract_fips,out_region,home,latitude,longitude
0,20000083,10000,-1,1,IL,USA,17,31,802403,0,1,42.141870,-87.953126
1,20000083,10002,1,2,IL,USA,17,31,801601,0,0,42.145289,-87.863023
2,20000083,10401,-1,4,IL,USA,17,31,801606,0,0,42.116325,-87.868708
3,20000083,10402,-1,4,IL,USA,17,31,803007,0,0,42.131743,-87.970674
4,20000083,10403,-1,4,IL,USA,17,31,802504,0,0,42.134326,-87.937060


In [4]:
data = pd.read_csv('./data/location.csv')


data = data.loc[(data.loctype == 1) | (data.loctype == 2) | (data.loctype == 3)]

v = data.sampno.value_counts()

data = data[data.sampno.apply(lambda x: v[x] > 1)]

data

,sampno,locno,perno,loctype,state,country,state_fips,county_fips,tract_fips,out_region,home,latitude,longitude
0,20000083,10000,-1,1,IL,USA,17,31,802403,0,1,42.141870,-87.953126
1,20000083,10002,1,2,IL,USA,17,31,801601,0,0,42.145289,-87.863023
11,20000136,10000,-1,1,IL,USA,17,31,81402,0,1,41.891022,-87.612931
12,20000136,10002,1,2,IL,USA,17,31,830900,0,0,41.928424,-87.684906
17,20000136,20002,2,2,IL,USA,17,31,81401,0,0,41.895035,-87.619717
...,...,...,...,...,...,...,...,...,...,...,...,...,...
110062,70100988,20003,2,3,IL,USA,17,31,281900,0,0,41.878635,-87.642924
110071,70100992,10000,-1,1,IL,USA,17,97,864519,0,1,42.165420,-87.959139
110072,70100992,10002,1,2,IL,USA,17,97,864205,0,0,42.292549,-88.141232
110074,70100993,10000,-1,1,IL,USA,17,31,260700,0,1,41.877056,-87.727996


In [18]:
import tqdm

In [31]:
tracts = gpd.read_file('osmnx/data/tracts.geojson')

tract_sampno = [None for i in range(len(data.index))]

for i in tqdm.tqdm(range(len(data.index))):
    s = data.iloc[i]
    for row in tracts.iloc:
        if row.geometry.contains(Point(s.longitude, s.latitude)):
            tract_sampno[i] = row.namelsad10
            break
            

data

100%|██████████| 32922/32922 [19:50<00:00, 27.66it/s]


,sampno,locno,perno,loctype,state,country,state_fips,county_fips,tract_fips,out_region,home,latitude,longitude,tract
0,20000083,10000,-1,1,IL,USA,17,31,802403,0,1,42.141870,-87.953126,0.0
1,20000083,10002,1,2,IL,USA,17,31,801601,0,0,42.145289,-87.863023,0.0
11,20000136,10000,-1,1,IL,USA,17,31,81402,0,1,41.891022,-87.612931,0.0
12,20000136,10002,1,2,IL,USA,17,31,830900,0,0,41.928424,-87.684906,0.0
17,20000136,20002,2,2,IL,USA,17,31,81401,0,0,41.895035,-87.619717,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110062,70100988,20003,2,3,IL,USA,17,31,281900,0,0,41.878635,-87.642924,0.0
110071,70100992,10000,-1,1,IL,USA,17,97,864519,0,1,42.165420,-87.959139,0.0
110072,70100992,10002,1,2,IL,USA,17,97,864205,0,0,42.292549,-88.141232,0.0
110074,70100993,10000,-1,1,IL,USA,17,31,260700,0,1,41.877056,-87.727996,0.0


In [ ]:
data['tract'] = tract_sampno

data.to_csv('./data/tract_location.csv')

In [45]:
filtered = pd.DataFrame(columns=data.columns)

for s in data.sampno.unique():
    snip = data.loc[data.sampno == s]
    if None in list(snip.tract): continue

    filtered = pd.concat([filtered, snip])

/tmp/ipykernel_126372/425088475.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  filtered = pd.concat([filtered, snip])


,sampno,locno,perno,loctype,state,country,state_fips,county_fips,tract_fips,out_region,home,latitude,longitude,tract
11,20000136,10000,-1,1,IL,USA,17,31,81402,0,1,41.891022,-87.612931,Census Tract 814.02
12,20000136,10002,1,2,IL,USA,17,31,830900,0,0,41.928424,-87.684906,Census Tract 8309
17,20000136,20002,2,2,IL,USA,17,31,81401,0,0,41.895035,-87.619717,Census Tract 814.01
35,20000300,10000,-1,1,IL,USA,17,31,81300,0,1,41.898335,-87.620753,Census Tract 813
36,20000300,10002,1,2,IL,USA,17,31,281900,0,0,41.878635,-87.642924,Census Tract 2819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110044,70100917,10000,-1,1,IL,USA,17,31,690300,0,1,41.768733,-87.629116,Census Tract 6903
110045,70100917,30003,3,3,IL,USA,17,31,690300,0,0,41.768733,-87.629116,Census Tract 6903
110060,70100988,10000,-1,1,IL,USA,17,31,834300,0,1,41.745835,-87.591788,Census Tract 8343
110061,70100988,10003,1,3,IL,USA,17,31,281900,0,0,41.878635,-87.642924,Census Tract 2819


In [66]:
sampnos = filtered.sampno.unique()

origin_dest_pairs = {}

for s in sampnos:
    origin = list(filtered.loc[(filtered.sampno == s) & (filtered.home == 1)].tract)[0]
    destinations = list(filtered.loc[(filtered.sampno == s) & (filtered.home == 0)].tract)
    print(destinations)
    origin_dest_pairs.setdefault(origin, [])

    for d in destinations:
        origin_dest_pairs[origin] += [d]

['Census Tract 8309', 'Census Tract 814.01']
['Census Tract 2819']
['Census Tract 8381', 'Census Tract 207.01']
['Census Tract 814.01', 'Census Tract 8431']
['Census Tract 8391']
['Census Tract 8391']
['Census Tract 7608.01']
['Census Tract 3201']
['Census Tract 815']
['Census Tract 8391']
['Census Tract 714']
['Census Tract 8391']
['Census Tract 8391']
['Census Tract 7201', 'Census Tract 3818']
['Census Tract 2819']
['Census Tract 2801']
['Census Tract 2420']
['Census Tract 8391']
['Census Tract 8391']
['Census Tract 3201']
['Census Tract 8391', 'Census Tract 817']
['Census Tract 814.01', 'Census Tract 3204']
['Census Tract 8391']
['Census Tract 1610']
['Census Tract 3201', 'Census Tract 8391']
['Census Tract 4204', 'Census Tract 208.02']
['Census Tract 8362', 'Census Tract 8362']
['Census Tract 818', 'Census Tract 814.01']
['Census Tract 814.03', 'Census Tract 501', 'Census Tract 602']
['Census Tract 5003']
['Census Tract 3204']
['Census Tract 8419']
['Census Tract 8411']
['Census Tr

In [70]:
import json

json.dump(origin_dest_pairs, open('data/origin_pairs.json', 'w'), indent=2)

In [7]:
import json

pairs = json.load(open('data/origin_pairs.json', 'r'))

results = {}

for origin, destinations in pairs.items():
    results[origin] = {d: destinations.count(d) for d in set(destinations)}


In [9]:
json.dump(results, open('data/origin_dest_weights.json', 'w'), indent=2)